In [1]:
import os
import json
import pandas as pd
from maomao.utils.constants import *
from maomao.parsing.metadata_utils import *
from maomao.parsing.integrated_dataset_utils import *

#### Cytolytic dataset integration and label-consistency analysis

- The input consists of a processed cytolytic dataset (processed_cytolytic_dataset.csv) containing peptide sequences and their corresponding cytolytic labels derived from AMPDB. Each sequence may appear multiple times across sources or with incomplete annotation coverage.

- The pipeline begins by extracting all unique peptide sequences and constructing a pivot table in which each row corresponds to a unique sequence and each column represents a data source. This pivot representation enables consistent aggregation and comparison of annotations at the sequence level.

- Quality control is applied in two stages. First, sequences containing non-canonical amino acids are removed. Second, sequences are filtered according to minimum and maximum length constraints defined in the global configuration. These steps ensure that all retained sequences are suitable for downstream machine learning and representation learning workflows.

- After filtering, source-specific cytolytic labels are mapped onto the pivot table using a standardized encoding scheme (positive, negative, unlabeled, unknown). Label consistency is then evaluated by computing per-sequence label counts, positive and negative vote percentages, and a set of high-level classification flags that identify exclusive positives, exclusive negatives, unlabeled-only sequences, and sequences with conflicting evidence.

- Sequences with inconsistent annotations are explicitly classified as ambiguous and further stratified according to the proportion of positive votes, allowing fine-grained control over ambiguity thresholds in downstream analyses.

- The notebook produces multiple non-overlapping sequence subsets, including: strictly cytolytic (only-positive), strictly non-cytolytic (only-negative), and ambiguous sequences with mixed evidence.

- In addition, a comprehensive metadata file is generated, summarizing filtering statistics, sequence length distributions, label agreement metrics, and dataset composition. All outputs are exported in a reproducible, analysis-ready format for downstream modeling, benchmarking, and dataset release.

In [2]:
name_task = "toxic_effect_classification"
output_folder = "../../processed_data/integrating_and_cleaning_data/cytolytic"

# PATH_EXPORT are imported from peptide_toxicity_classifier.constants.
# Update them in constants.py according to the required input and export paths.

- Reading all sources

In [3]:
df_AMPDB_cytolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/AMPDB v1/processed_cytolytic_dataset.csv")
df_AMPDB_cytolytic = df_AMPDB_cytolytic.rename(columns={"label": "cytolytic"})

In [4]:
df_DRAMP_cytolytic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/DRAMP/processed_cytolytic_dataset.csv")
df_DRAMP_cytolytic = df_DRAMP_cytolytic.rename(columns={"label": "cytolytic"})

- Collecting all sequences for activity

In [5]:
df_list_cytolytic = [df_AMPDB_cytolytic, df_DRAMP_cytolytic]
unique_sequence = count_unique_sequence(df_list_cytolytic)

120


- Create pivote dataset

In [6]:
df_pivote = create_pivote(unique_sequence)

- Removing sequences with non canonical residues 

In [7]:
n_before_canon = df_pivote.shape[0]
df_pivote["is_canon"] = df_pivote["sequence"].apply(check_sequence)
n_after_canon = df_pivote[df_pivote["is_canon"]].shape[0]

In [8]:
print(df_pivote["is_canon"].value_counts())
df_pivote = df_pivote[df_pivote["is_canon"]]

is_canon
True     106
False     14
Name: count, dtype: int64


- Filter sequences by length

In [9]:
df_pivote["length"] = df_pivote["sequence"].str.len()
df_pivote["length"].describe()

count    106.000000
mean      74.462264
std       60.954186
min        5.000000
25%       26.000000
50%       69.000000
75%       95.750000
max      432.000000
Name: length, dtype: float64

In [10]:
n_before_length = n_after_canon
df_pivote["filter_length"] = df_pivote["length"].apply(check_length)
n_after_length = df_pivote[df_pivote["filter_length"]].shape[0]

In [11]:
df_pivote["filter_length"].value_counts()

filter_length
True     60
False    46
Name: count, dtype: int64

In [12]:
length_series = df_pivote[df_pivote["filter_length"]]["length"]

length_dist = {
    "min": length_series.min(),
    "max": length_series.max(),
    "mean": length_series.mean(),
    "median": length_series.median()
}

In [13]:
df_pivote = df_pivote[df_pivote["filter_length"]]
df_pivote.shape

(60, 4)

In [14]:
df_pivote = df_pivote.drop(columns=["is_canon", "filter_length", "length"])

In [15]:
df_list = [("AMPDB", df_AMPDB_cytolytic),
           ("DRAMP", df_DRAMP_cytolytic)]

In [16]:
for source, dataset in df_list:
    dataset = dataset[["sequence", "cytolytic"]]
    dataset = dataset.drop_duplicates(subset="sequence")
    mapping = dataset.set_index("sequence")["cytolytic"]

    # Mapear sin explotar memoria
    df_pivote[source] = (
        df_pivote["sequence"]
        .map(mapping)
        .fillna(999)
        .astype("int16")
    )

In [17]:
df_pivote.head()

,sequence,AMPDB,DRAMP
1,GFFGNTWKKIKGKADKIMLKKAVKIMVKKEGISKEEAQAKVDAMSK...,999,1
3,GFFGNTWKKIKGKADKIMLKKAVKIMVKKEGISKEEAQAKVDAMSK...,999,1
4,GFFGNTWKKIKGKADKIMLKKAVKIMVKKEGISKEEAQAKVDAMSK...,999,1
6,IWLTALKFLGKNLGKHLAKQQLAKL,999,1
14,INWKALLDAAKKVL,999,1


- Working with pivote for detecting ambiguous sequences 

In [18]:
df_pivote = process_count_labels(df_pivote) # Verify the consistency of the labels by source

In [19]:
df_pivote["negative"].value_counts() # Includes sources labeled as nevative (0) and unlabeled (2)

negative
False    58
True      2
Name: count, dtype: int64

In [20]:
df_pivote["exclusive_0"].value_counts()

exclusive_0
False    58
True      2
Name: count, dtype: int64

In [21]:
df_pivote["positive"].value_counts() # Includes sources labeled as positive (1) and unlabeled (2)

positive
True     58
False     2
Name: count, dtype: int64

In [22]:
df_pivote["exclusive_1"].value_counts()

exclusive_1
True     58
False     2
Name: count, dtype: int64

In [23]:
df_pivote["only_unlabel"].value_counts() # Includes only sources unlabeled (2)

only_unlabel
False    60
Name: count, dtype: int64

In [24]:
df_pivote.sort_values(by="percentage_1", ascending=False)

,sequence,AMPDB,DRAMP,counts_1,counts_0,counts_unlabel,counts_unknown,positive,negative,exclusive_1,exclusive_0,only_unlabel,percentage_0,percentage_1
1,GFFGNTWKKIKGKADKIMLKKAVKIMVKKEGISKEEAQAKVDAMSK...,999,1,1,0,0,1,True,False,True,False,False,0.0,100.0
3,GFFGNTWKKIKGKADKIMLKKAVKIMVKKEGISKEEAQAKVDAMSK...,999,1,1,0,0,1,True,False,True,False,False,0.0,100.0
4,GFFGNTWKKIKGKADKIMLKKAVKIMVKKEGISKEEAQAKVDAMSK...,999,1,1,0,0,1,True,False,True,False,False,0.0,100.0
6,IWLTALKFLGKNLGKHLAKQQLAKL,999,1,1,0,0,1,True,False,True,False,False,0.0,100.0
14,INWKALLDAAKKVL,999,1,1,0,0,1,True,False,True,False,False,0.0,100.0
15,GMKDYFKKLLQKVIKKIKSFRKKQD,1,999,1,0,0,1,True,False,True,False,False,0.0,100.0
18,GILDTIKSIASKVWNSKTVQDLKRKGINWVANKLGVSPQAA,999,1,1,0,0,1,True,False,True,False,False,0.0,100.0
20,FHPSLWVLIPQYIQLIRKILKSG,1,999,1,0,0,1,True,False,True,False,False,0.0,100.0
23,GFFALIPKIISSPIFKTLLSAVGSALSSSGGQE,999,1,1,0,0,1,True,False,True,False,False,0.0,100.0
22,QIVDCWETWSRCTKWSQGGTGTLWKSCNDRCKELGRKRGQCEEKPS...,999,1,1,0,0,1,True,False,True,False,False,0.0,100.0


- Splitting data into only negative, only positive, and with amiguous data

In [25]:
negative = df_pivote[df_pivote["negative"]]

In [26]:
only_negative = df_pivote[df_pivote["exclusive_0"]]

In [27]:
positive = df_pivote[df_pivote["positive"]]

In [28]:
only_positive = df_pivote[df_pivote["exclusive_1"]]

In [29]:
only_unlabel = df_pivote[df_pivote["only_unlabel"]]

In [30]:
df_ambiguous = df_pivote[(df_pivote["positive"] == False) & (df_pivote["negative"] == False) & (df_pivote["only_unlabel"] == False)]

- Processing ambiguous data

In [31]:
df_ambiguous = categorize_percentage(df_ambiguous)

In [32]:
df_ambiguous["Category_pbb"].value_counts()

Series([], Name: count, dtype: int64)

- Working with metada

In [33]:
seq_stats = {
    "canonical": {
        "before": n_before_canon,
        "after": n_after_canon
    },
    "length": {
        "before": n_before_length,
        "after": n_after_length,
        "min": MIN_LENGTH_SEQUENCE,
        "max": MAX_LENGTH_SEQUENCE
    },
    "length_dist": length_dist
}

metadata = build_dataset_metadata(
    task="cytolytic",
    source_list=df_list,
    pivote_df=df_pivote,
    outputs={
        "only_positive": only_positive,
        "only_negative": only_negative,
        "ambiguous": df_ambiguous
    },
    seq_stats=seq_stats,
    filters={
        "canonical_residues": True,
        "length_filter": True
    }
)
metadata

{'task': 'cytolytic',
 'generated_at': '2026-07-24T16:23:12.201493',
 'sources': {'n_unique_sequences': {'AMPDB': 52, 'DRAMP': 69}},
 'filters': {'canonical_residues': {'applied': True},
  'length_filter': {'applied': True, 'min_length': 5, 'max_length': 70}},
 'sequence_statistics': {'canonical_filter': {'before': 120, 'after': 106},
  'length_filter': {'before': 106, 'after': 60},
  'length_distribution': {'min': 5, 'max': 70, 'mean': 37.22, 'median': 30.0}},
 'statistics': {'total_sequences_final': 60,
  'positive': {'positive_and_unlabel': 58, 'only_positive': 58},
  'negative': {'negative_and_unlabel': 2, 'only_negative': 2},
  'only_unlabel': 0,
  'ambiguous': {'n_sequences': 0}}}

- Exporting data

In [34]:
os.makedirs(output_folder, exist_ok=True)

In [35]:
with open(f"{output_folder}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [36]:
positive.shape

(58, 14)

In [37]:
only_positive.shape

(58, 14)

In [38]:
negative.shape

(2, 14)

In [39]:
only_negative.shape

(2, 14)

In [40]:
only_unlabel.shape

(0, 14)

In [41]:
df_ambiguous.shape

(0, 15)

In [42]:
positive.to_csv(f"{output_folder}/positive.csv", index=False)

In [43]:
negative.to_csv(f"{output_folder}/negative.csv", index=False)